In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering, AffinityPropagation, KMeans
from sklearn.mixture import GaussianMixture
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

In [17]:
group_annotations = pd.read_csv('../data/processed/group_annotation.csv', index_col=0)
expression_data = pd.read_csv('expression_data.csv')
gene_variances = expression_data.var(axis=1)

top_n_list = [10, 100, 1000, 5000, 10000]
k = 3
agglo_labels_dict, kmeans_labels_dict, gmm_labels_dict, ap_labels_dict = {}, {}, {}, {}

agglo = AgglomerativeClustering(n_clusters=k, compute_distances=True)
kmeans = KMeans(n_clusters=k, random_state=27)
gmm = GaussianMixture(n_components=k, random_state=42, covariance_type='full')
ap = AffinityPropagation(affinity='euclidean', random_state=3)

for n in top_n_list:
    top_variable_genes = gene_variances.sort_values(ascending=False).head(n).index
    top_variable_expression = expression_data.loc[top_variable_genes]
    group_annotations_n = group_annotations.loc[top_variable_expression.T.index]
    data_for_clustering = top_variable_expression.T

    agglo_labels_dict[n] = agglo.fit_predict(data_for_clustering)

    kmeans_labels_dict[n] = kmeans.fit_predict(data_for_clustering)

    gmm_labels_dict[n] = gmm.fit_predict(data_for_clustering)

    X_pca = PCA(n_components=2, random_state=42).fit_transform(data_for_clustering)
    similarity = -pairwise_distances(X_pca, metric='euclidean')
    ap_labels_dict[n] = ap.fit_predict(similarity)

In [13]:
def chi_squared_clusters_vs_surgery(cluster_labels):
   contingency_table = pd.crosstab(cluster_labels, group_annotations['Group']=='T0')
   chi2, p, _, _ = chi2_contingency(contingency_table)
   #res = pd.DataFrame({'Genes 1': n1, 'Genes 2': n2, 'Chi2 Statistic': chi2, 'p-value': p})
  
   return chi2, p

Adjustment for Multiple Hypothesis Testing

In [15]:
results = []
clusterings = {
    'Agglo': agglo_labels_dict,
    'KMeans': kmeans_labels_dict,
    'GMM': gmm_labels_dict,
    'AP': ap_labels_dict
}

for name, labels_dict in clusterings.items():
    for i, n1 in enumerate(top_n_list):
        for n2 in top_n_list[i+1:]:
            contingency = pd.crosstab(labels_dict[n1], labels_dict[n2])
            chi2, p, _, _ = chi2_contingency(contingency)
            results.append({'Method': name, 'Group 1': n1, 'Group 2': n2, 'Chi2 Statistic': chi2, 'p-value': p})

    for num_genes, labels in labels_dict.items():
        chi2, p = chi_squared_clusters_vs_surgery(labels)
        results.append({'Method': name, 'Group 1': num_genes, 'Chi2 Statistic': chi2, 'p-value': p})

results_df = pd.DataFrame(results)

# Adjust p-values for multiple testing (Benjamini-Hochberg FDR)
results_df['adj_p-value'] = multipletests(results_df['p-value'], method='fdr_bh')[1]

print(results_df)

    Method  Group 1  Group 2  Chi2 Statistic        p-value    adj_p-value
0    Agglo       10    100.0       77.412500   6.151033e-16   1.845310e-15
1    Agglo       10   1000.0       25.105633   4.790829e-05   5.988537e-05
2    Agglo       10   5000.0       17.591775   1.482649e-03   1.617435e-03
3    Agglo       10  10000.0       18.919145   8.152181e-04   9.057978e-04
4    Agglo      100   1000.0       56.532143   1.550835e-11   4.045656e-11
5    Agglo      100   5000.0       52.222222   1.239470e-10   2.564420e-10
6    Agglo      100  10000.0       48.051151   9.208931e-10   1.726675e-09
7    Agglo     1000   5000.0      131.022222   2.353857e-27   9.415429e-27
8    Agglo     1000  10000.0      122.132554   1.871150e-25   7.016812e-25
9    Agglo     5000  10000.0      167.497585   3.601597e-35   2.701198e-34
10   Agglo       10      NaN       19.698701   5.278146e-05   6.463035e-05
11   Agglo      100      NaN        5.000000   8.208500e-02   8.794821e-02
12   Agglo     1000      